# Hi-res 1601² CFDAC — GPU run (deep, resolution-appropriate net + long training)

**Why this notebook exists.** The CPU preliminary in `results_hires/` trained the *top 128² cell* per task (a 4-layer `Conv2DStack`, or a timm backbone that resizes to 224) on 1601² CFDAC for only 4 epochs. Those architectures **throw the resolution away** (a strided stem + global-average-pool, or a 224 resize) and are under-trained — so they cannot answer *“does full resolution help?”* fairly.

**What this does.** On a GPU it trains a genuinely deep CNN (`DeepCFDACNet`, ResNet18-style, ~11M params) that **consumes the full 1601² grid** (7 downsampling stages, no 224 resize), with **many epochs + cosine LR schedule + class weights + best-val checkpoint**, on full-resolution CFDAC computed **on the GPU** from the raw FRFs (no caching needed). It evaluates each of the 10 tasks on held-out synth (in-domain) and the 2 638-case experimental set (zero-shot), reporting **balanced accuracy / macro-F1** (not raw accuracy) and flagging class-collapse.

Results are written to `results_hires_v2/` with the **same schema** as the CPU run so they drop straight into `ml_pipeline/hires_summary.py` and `REPORT_CONSOLIDATED.md`.

**Runtime.** Set a GPU runtime (Runtime → Change runtime type → GPU; A100/V100 ideal, T4 works). Defaults below do all 10 tasks; per-task skip-if-exists lets you resume across Colab sessions. Lower `BATCH` if you hit CUDA OOM.


## 1 · Bootstrap — clone repos (phd_lanl + pymodal as siblings) and install deps

In [ ]:
import os, sys, subprocess
GH_USER = 'grcarmenaty'
WORK = '/content'
os.chdir(WORK)

def _gh_token():
    # Optional: store a GitHub PAT as a Colab secret named GH_TOKEN if the repos are private.
    try:
        from google.colab import userdata
        return userdata.get('GH_TOKEN')
    except Exception:
        return os.environ.get('GH_TOKEN')

def clone(repo, branch):
    dst = os.path.join(WORK, repo if repo != 'phd_lanl' else 'PhD_LANL')
    if os.path.isdir(dst):
        print('exists:', dst); return dst
    tok = _gh_token()
    auth = f'{tok}@' if tok else ''
    url = f'https://{auth}github.com/{GH_USER}/{repo}.git'
    r = subprocess.run(['git','clone','--depth','1','-b',branch,url,dst])
    if r.returncode != 0:
        raise RuntimeError(f'clone failed for {repo}@{branch} (private? set a GH_TOKEN Colab secret)')
    return dst

# phd_lanl on main (where the hi-res work lives); pymodal on master (sibling dir the scripts expect).
REPO   = clone('phd_lanl', 'main')
PYMOD  = clone('pymodal',  'master')
# scripts look for pymodal at _REPO.parent/'pymodal' -> make a sibling named exactly 'pymodal'
if os.path.basename(PYMOD) != 'pymodal':
    os.symlink(PYMOD, os.path.join(WORK, 'pymodal'))
for p in (REPO, os.path.join(WORK,'pymodal')):
    if p not in sys.path: sys.path.insert(0, p)
os.chdir(REPO)
print('cwd =', os.getcwd())
subprocess.run([sys.executable,'-m','pip','-q','install','timm','h5py','scikit-learn','pint','pyFRF','audiomentations'])
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — set a GPU runtime!'))


## 2 · Regenerate the 1601-bin features (gitignored, so rebuilt from committed sources)

Same steps the CPU run used: regenerate the synth dataset at `N_T=4096, fs=256` (16 s sim → df=0.0625 Hz → 1601 bins), build `features_hires.h5`; reassemble `experimental_frfs.h5` from its git chunks, derive labels via `primary_op`, build `experimental_features_hires.h5`. ~3 min total.

In [ ]:
import subprocess, sys, os, glob, json, h5py, numpy as np
from pathlib import Path
REPO = Path(os.getcwd())

def run(cmd):
    print('>>', ' '.join(cmd)); r = subprocess.run(cmd); assert r.returncode==0, cmd

# 2a. synth chunks @ N_T=4096 + synth hi-res features
if not (REPO/'dataset'/'features_hires.h5').exists():
    run([sys.executable,'ml_pipeline/generate_dataset.py','--out','dataset_hires','--n-t','4096','--fs','256'])
    run([sys.executable,'ml_pipeline/build_hires_synth_features.py'])

# 2b. reassemble experimental_frfs.h5 from committed chunks
if not (REPO/'experimental_frfs.h5').exists():
    parts = sorted(glob.glob('experimental_frfs_chunks/experimental_frfs.h5.part_*'))
    with open('experimental_frfs.h5','wb') as out:
        for p in parts:
            with open(p,'rb') as f: out.write(f.read())

# 2c. minimal exp label file (names + decomposed labels via the same primary_op parser)
if not (REPO/'dataset'/'experimental_features.h5').exists():
    from ml_pipeline.evaluate import primary_op
    with h5py.File('experimental_frfs.h5','r') as f:
        names = json.loads(f.attrs['case_names_json'])
    n=len(names); tc=np.zeros(n,np.int8); st=np.full(n,-1,np.int8); en=np.full(n,-1,np.int8); sv=np.zeros(n,np.float32)
    for i,nm in enumerate(names):
        op=primary_op(nm); tc[i]=op['type_code']; st[i]=op['storey']; en[i]=op['end']; sv[i]=op['severity']
    dt=h5py.string_dtype('utf-8')
    with h5py.File('dataset/experimental_features.h5','w') as o:
        o.create_dataset('names',data=np.array(names,dtype=object),dtype=dt)
        o.create_dataset('type_code',data=tc); o.create_dataset('storey',data=st)
        o.create_dataset('end',data=en); o.create_dataset('severity',data=sv)

# 2d. exp hi-res features
if not (REPO/'dataset'/'experimental_features_hires.h5').exists():
    run([sys.executable,'ml_pipeline/build_hires_exp_features.py'])

print('features ready:',
      (REPO/'dataset'/'features_hires.h5').exists(),
      (REPO/'dataset'/'experimental_features_hires.h5').exists())


## 3 · GPU CFDAC, the deep resolution-appropriate network, and a lazy dataset

CFDAC is computed **on the GPU** per batch straight from the complex FRFs (the exact `pymodal.value_CFDAC` formula, in torch). `DeepCFDACNet` is a ResNet18-style stack with a stride-2 stem + maxpool and 4 residual stages (64→128→256→512), so 1601² is processed through real receptive fields down to ~13² before global pooling — no 224 resize, no premature global-pool of a near-full-res map.

In [ ]:
import torch, torch.nn as nn, numpy as np, h5py
from torch.utils.data import Dataset, DataLoader
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def cfdac_torch(ref, frf):
    """ref:(N,9) complex shared; frf:(B,N,9) complex -> (B,2,N,N) float32 (real,imag), per-sample mean-subtracted.
    Matches pymodal.utils.value_CFDAC: C[i,j]=(frf[i]·conj(ref[j]))^2 / (||frf[i]||^2 ||ref[j]||^2)."""
    with torch.no_grad():
        cross = torch.matmul(frf, ref.conj().t())              # (B,N,N)
        num   = cross**2
        nf    = (frf.abs()**2).sum(-1)                          # (B,N)
        nr    = (ref.abs()**2).sum(-1)                          # (N,)
        denom = nf.unsqueeze(-1) * nr.unsqueeze(0).unsqueeze(0) # (B,N,N)
        C = torch.nan_to_num(num / denom)
        out = torch.stack([C.real, C.imag], dim=1).float()      # (B,2,N,N)
        out = out - out.mean(dim=(-2,-1), keepdim=True)         # per-sample mean-subtract (real & imag)
    return out

class FRFDataset(Dataset):
    """Holds complex FRFs in RAM; returns raw complex FRF per sample. CFDAC is done on-GPU in the loop."""
    def __init__(self, H, y):
        self.H = torch.from_numpy(H.astype(np.complex64)); self.y = y
    def __len__(self): return len(self.H)
    def __getitem__(self, i): return self.H[i], self.y[i]

def conv_bn(ci,co,k=3,s=1,p=1): return nn.Sequential(nn.Conv2d(ci,co,k,s,p,bias=False), nn.BatchNorm2d(co))
class BasicBlock(nn.Module):
    def __init__(self, ci, co, stride=1):
        super().__init__()
        self.c1=conv_bn(ci,co,3,stride,1); self.c2=conv_bn(co,co,3,1,1)
        self.act=nn.GELU()
        self.down = conv_bn(ci,co,1,stride,0) if (stride!=1 or ci!=co) else None
    def forward(self,x):
        idt = x if self.down is None else self.down(x)
        x = self.act(self.c1(x)); x = self.c2(x)
        return self.act(x+idt)

class DeepCFDACNet(nn.Module):
    """ResNet18-style net that consumes the full 1601² grid."""
    def __init__(self, n_in=2, n_out=2, widths=(64,128,256,512), regression=False):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(n_in,widths[0],7,2,3,bias=False),
                                  nn.BatchNorm2d(widths[0]), nn.GELU(),
                                  nn.MaxPool2d(3,2,1))            # 1601 -> 801 -> 401
        c=widths[0]; stages=[]
        for i,w in enumerate(widths):
            s = 1 if i==0 else 2
            stages += [BasicBlock(c,w,s), BasicBlock(w,w,1)]; c=w   # 401->401->201->101->51
        self.stages = nn.Sequential(*stages)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Linear(c,128), nn.GELU(), nn.Dropout(0.3),
                                  nn.Linear(128,n_out))
        self.regression=regression
    def forward(self,x): return self.head(self.stages(self.stem(x)))
print('model + GPU CFDAC defined on', DEV)


## 4 · Config, train/eval, and the 10-task sweep

Tune `EPOCHS`, `SUBSAMPLE`, `BATCH` for your GPU/time budget. Per-task results are written and skipped on resume.

In [ ]:
import time, json, numpy as np, torch, h5py
from pathlib import Path
from sklearn.metrics import (accuracy_score, f1_score, r2_score,
                             balanced_accuracy_score)
from ml_pipeline.tasks import build_targets
from ml_pipeline.train import make_split

# ---- knobs ----
SEED      = 42
SUBSAMPLE = 3000     # synth samples per task (raise toward 10000 on a fast GPU)
EPOCHS    = 25       # cosine-scheduled
BATCH     = 16       # lower to 8 if CUDA OOM
LR        = 3e-4
WARMUP    = 2
TASKS     = ['binary','col_location','mass_location','severity','type',
             'is_bolt','is_crack','is_mass','is_hole','is_pristine']
OUT = Path('results_hires_v2'); (OUT/'per_case').mkdir(parents=True, exist_ok=True)
(OUT/'models').mkdir(exist_ok=True)
torch.manual_seed(SEED); np.random.seed(SEED)

SYN = Path('dataset/features_hires.h5'); EXP = Path('dataset/experimental_features_hires.h5')
with h5py.File(SYN,'r') as f:
    syn_tasks = build_targets(f['type_code'][:].astype(np.int64), f['storey'][:].astype(np.int64),
                              f['end'][:].astype(np.int64), f['severity'][:].astype(np.float32))
    H_ref_syn = torch.from_numpy(f['reference/frf_complex'][:].astype(np.complex64)).to(DEV)
with h5py.File(EXP,'r') as f:
    exp_tasks = build_targets(f['type_code'][:].astype(np.int64), f['storey'][:].astype(np.int64),
                              f['end'][:].astype(np.int64), f['severity'][:].astype(np.float32))
    names = [str(s) for s in f['names'][:]]
    H_ref_exp = torch.from_numpy(f['reference/frf_complex'][:].astype(np.complex64)).to(DEV)
print('exp FRFs loading...');
with h5py.File(EXP,'r') as f:
    H_exp = (f['frf_real'][:] + 1j*f['frf_imag'][:]).astype(np.complex64)
print('exp FRFs', H_exp.shape)

def _predict(mdl, H, ref, kind, bs=BATCH):
    mdl.eval(); preds=[]; probs=[]
    with torch.no_grad():
        for i in range(0, len(H), bs):
            fb = torch.from_numpy(H[i:i+bs]).to(DEV)
            x  = cfdac_torch(ref, fb)
            with torch.autocast('cuda', enabled=(DEV.type=='cuda')):
                out = mdl(x).float().cpu().numpy()
            if kind=='cls':
                preds.append(out.argmax(1))
                e=np.exp(out-out.max(1,keepdims=True)); probs.append(e/e.sum(1,keepdims=True))
            else:
                preds.append(out.squeeze(1) if out.ndim==2 else out)
    return np.concatenate(preds), (np.concatenate(probs) if kind=='cls' else None)

def train_task(task):
    tag=f'{task}_deepcfdac_cfdac_realimag_hires1601'
    pc=OUT/'per_case'/f'{tag}.json'
    if pc.exists(): print('skip',tag,'(exists)'); return
    mask,y_all,kind = syn_tasks[task]
    idx_pool=np.where(mask)[0]; rng=np.random.default_rng(20260518+SEED)
    keep=np.sort(rng.choice(len(idx_pool), size=min(SUBSAMPLE,len(idx_pool)), replace=False))
    rows_sel=np.sort(idx_pool[keep]); y=y_all[keep][np.argsort(idx_pool[keep].argsort())]
    # load FRFs for the subsample (sorted), restore order
    with h5py.File(SYN,'r') as f:
        order=np.argsort(idx_pool[keep]); sr=idx_pool[keep][order]
        re=f['frf_real'][sr]; im=f['frf_imag'][sr]
    H=np.empty_like(re,dtype=np.complex64); H[order]=(re+1j*im).astype(np.complex64)
    y=y_all[keep]
    i_tr,i_va,i_te=make_split(y,kind); n_out=(int(y.max())+1) if kind=='cls' else 1
    cls_w=None
    if kind=='cls':
        cnt=np.bincount(y[i_tr],minlength=n_out).astype(np.float32)
        cls_w=torch.tensor(cnt.sum()/np.clip(cnt*n_out,1e-6,None)).float().to(DEV)
    mdl=DeepCFDACNet(2,n_out,regression=(kind=='reg')).to(DEV)
    opt=torch.optim.AdamW(mdl.parameters(), lr=LR, weight_decay=1e-4)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1,EPOCHS-WARMUP))
    lossf=nn.CrossEntropyLoss(weight=cls_w) if kind=='cls' else nn.MSELoss()
    scaler=torch.cuda.amp.GradScaler(enabled=(DEV.type=='cuda'))
    def batched(ii): 
        for j in range(0,len(ii),BATCH): yield ii[j:j+BATCH]
    best=-1e9; best_state=None; t0=time.time()
    for ep in range(EPOCHS):
        mdl.train(); perm=np.random.permutation(i_tr)
        for bi in batched(perm):
            fb=torch.from_numpy(H[bi]).to(DEV); x=cfdac_torch(H_ref_syn, fb)
            yb=torch.from_numpy(y[bi]).to(DEV)
            opt.zero_grad()
            with torch.autocast('cuda', enabled=(DEV.type=='cuda')):
                out=mdl(x)
                loss=lossf(out, yb.long()) if kind=='cls' else lossf(out, yb.float().unsqueeze(1))
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        if ep>=WARMUP: sched.step()
        pv,_=_predict(mdl, H[i_va], H_ref_syn, kind)
        m = f1_score(y[i_va],pv,labels=list(range(n_out)),average='macro',zero_division=0) if kind=='cls' else r2_score(y[i_va],pv)
        print(f'  {task} ep{ep+1}/{EPOCHS} val={m:+.4f} ({time.time()-t0:.0f}s)')
        if m>best: best=float(m); best_state={k:v.detach().clone() for k,v in mdl.state_dict().items()}
    if best_state: mdl.load_state_dict(best_state)
    torch.save(best_state, OUT/'models'/f'{tag}.pt')
    # synth test
    pt,_=_predict(mdl, H[i_te], H_ref_syn, kind)
    if kind=='cls':
        s_acc=accuracy_score(y[i_te],pt); s_mf1=f1_score(y[i_te],pt,labels=list(range(n_out)),average='macro',zero_division=0)
        s_bal=balanced_accuracy_score(y[i_te],pt)
    else:
        s_acc=r2_score(y[i_te],pt); s_mf1=None; s_bal=None
    # exp zero-shot
    mask_e,e_y,e_kind=exp_tasks[task]; idx_e=np.where(mask_e)[0]
    pe,prob=_predict(mdl, H_exp[idx_e], H_ref_exp, e_kind)
    rows=[]
    for i,ix in enumerate(idx_e):
        r={'case':names[ix],'y_true':(int(e_y[i]) if e_kind=='cls' else float(e_y[i])),
           'y_pred':(int(pe[i]) if e_kind=='cls' else float(pe[i]))}
        if prob is not None: r['proba']=[float(p) for p in prob[i]]
        rows.append(r)
    meta={'task':task,'model':'deepcfdac','feature':'cfdac_realimag','n_target':1601,
          'kind':e_kind,'n_out':int(n_out),'epochs':EPOCHS,'subsample':int(min(SUBSAMPLE,len(idx_pool))),
          'synth_val_best':best,'synth_test_metric':float(s_acc),
          'synth_test_macro_f1':(float(s_mf1) if s_mf1 is not None else None),
          'synth_test_bal_acc':(float(s_bal) if s_bal is not None else None),
          'input_normalized':True}
    pc.write_text(json.dumps({'meta':meta,'rows':rows}, indent=2))
    # accumulate synth summary
    sp=OUT/'synth_test_v2.json'; allr=json.loads(sp.read_text()) if sp.exists() else {}
    allr[tag]=meta; sp.write_text(json.dumps(allr, indent=2))
    print(f'  DONE {tag}: synth mF1/R2={s_mf1 if s_mf1 is not None else s_acc:.3f}  wrote {pc.name}')

for t in TASKS:
    print('==',t,'=='); train_task(t)
print('\nsweep complete')


## 5 · Honest summary (balanced-acc / macro-F1 / collapse flags) + download

Mirrors `ml_pipeline/hires_summary.py`. Then zip `results_hires_v2/` for download (and there's an optional git-push cell).

In [ ]:
import json, numpy as np
from pathlib import Path
from collections import Counter
from sklearn.metrics import balanced_accuracy_score, f1_score, accuracy_score
OUT=Path('results_hires_v2')
print(f"{'cell':<42}{'kind':>5}{'synth':>9}{'expMF1/R2':>11}{'expBal':>8}{'expAcc':>8}{'prior':>7}{'collapse':>9}")
print('-'*101)
for p in sorted((OUT/'per_case').glob('*_hires1601.json')):
    d=json.loads(p.read_text()); m=d['meta']; r=d['rows']
    yt=np.array([x['y_true'] for x in r]); yp=np.array([x['y_pred'] for x in r])
    if m['kind']=='cls':
        n=m['n_out']; bal=balanced_accuracy_score(yt,yp)
        mf1=f1_score(yt,yp,labels=list(range(n)),average='macro',zero_division=0); acc=accuracy_score(yt,yp)
        prior=max(Counter(yt.tolist()).values())/len(yt); coll=(len(set(yp.tolist()))<=1) or (bal<=1/n+0.02)
        syn=m.get('synth_test_macro_f1') or 0
        print(f"{m['task']:<42}{'cls':>5}{syn:>9.3f}{mf1:>11.3f}{bal:>8.3f}{acc:>8.3f}{prior:>7.3f}{str(coll):>9}")
    else:
        yt=yt.astype(float); yp=yp.astype(float); mae=np.mean(np.abs(yt-yp))
        ss=np.sum((yt-yp)**2); st=np.sum((yt-yt.mean())**2); r2=1-ss/st if st>0 else 0
        print(f"{m['task']:<42}{'reg':>5}{(m.get('synth_test_metric') or 0):>9.3f}{r2:>11.3f}{'-':>8}{'-':>8}{'-':>7}{'MAE=%.3f'%mae:>9}")

import shutil
shutil.make_archive('results_hires_v2','zip','results_hires_v2')
try:
    from google.colab import files; files.download('results_hires_v2.zip')
except Exception as e:
    print('download manually: results_hires_v2.zip', e)


## 6 · (Optional) push results back to the repo

Requires a `GH_TOKEN` Colab secret with push rights. Commits only the JSONs (model weights stay local/gitignored).

In [ ]:
import os, subprocess
tok = None
try:
    from google.colab import userdata; tok = userdata.get('GH_TOKEN')
except Exception: tok = os.environ.get('GH_TOKEN')
if not tok:
    print('No GH_TOKEN — skip. Use the zip from cell 5 and hand it to the agent instead.')
else:
    subprocess.run(['git','config','user.email','colab@gpu.run'])
    subprocess.run(['git','config','user.name','colab-gpu'])
    subprocess.run(['git','add','results_hires_v2/per_case','results_hires_v2/synth_test_v2.json'])
    subprocess.run(['git','commit','-m','hires v2 (GPU): deep DeepCFDACNet @1601, long-train synth+exp'])
    url=f'https://{tok}@github.com/grcarmenaty/phd_lanl.git'
    print(subprocess.run(['git','push',url,'HEAD:main'], capture_output=True, text=True).stderr[-400:])
